# Battery Thermal Surrogate: Training Pipeline
Train a Physics-Conditioned U-Net to predict temperature field evolution. Demonstrates 3-phase training with physics-informed losses.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.utils.device import get_device
from src.utils.config import load_config, DEFAULTS
from src.utils.logging import TrainingLogger
from src.models.pc_unet import PCUNet
from src.models.simple_cnn import SimpleCNN
from src.training.losses import PhysicsInformedLoss
from src.training.gradnorm import GradNorm
from src.physics.solver import HeatSolver2D
from src.physics.materials import create_material_mask, compute_signed_distance

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

device = get_device()
print(f"Using device: {device}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Generate Training Data
Create a small synthetic dataset for demonstration. In production, use `scripts/generate_data.py` for the full dataset.

In [ ]:
# Grid and simulation parameters
grid_size = 32
n_trajectories = 50
n_steps = 50
dx = dy = 1e-3  # 1 mm grid spacing
dt = 5e-4       # time step chosen for stability

# Create fixed geometry: material mask and signed distance fields
mask = create_material_mask(grid_size=grid_size)
mask_cell = (mask == 0).astype(np.float64)
mask_coolant = (mask == 1).astype(np.float64)
mask_insulation = (mask == 2).astype(np.float64)
sdf_cell = compute_signed_distance(mask, material_id=0)
sdf_coolant = compute_signed_distance(mask, material_id=1)

# Normalise SDFs to [-1, 1] range
sdf_cell = sdf_cell / (np.abs(sdf_cell).max() + 1e-8)
sdf_coolant = sdf_coolant / (np.abs(sdf_coolant).max() + 1e-8)

print(f"Grid size: {grid_size}x{grid_size}")
print(f"Material distribution: cell={mask_cell.sum():.0f}, "
      f"coolant={mask_coolant.sum():.0f}, insulation={mask_insulation.sum():.0f}")

# Generate trajectories with random physical parameters
all_inputs = []   # will hold (N_samples, 9, H, W)
all_targets = []  # will hold (N_samples, 1, H, W)

for traj_i in range(n_trajectories):
    # Random physical parameters within realistic ranges
    k_val = np.random.uniform(1.0, 5.0)        # thermal conductivity [W/(m*K)]
    q0 = np.random.uniform(1e5, 5e6)            # heat source [W/m^3]
    h_conv = np.random.uniform(10.0, 500.0)      # convection coeff [W/(m^2*K)]
    T_amb = np.random.uniform(290.0, 310.0)      # ambient temp [K]
    T_init_val = T_amb + np.random.uniform(0.0, 5.0)  # initial temp offset

    # Create solver with uniform k for simplicity
    solver = HeatSolver2D(
        nx=grid_size, ny=grid_size, dx=dx, dy=dy, dt=dt,
        k=k_val, rho=2500.0, cp=700.0, T_amb=T_amb, h_conv=h_conv,
    )

    # Ensure stability
    if not solver.check_stability():
        # If unstable, reduce dt to max stable value
        dt_safe = solver.max_stable_dt * 0.9
        solver = HeatSolver2D(
            nx=grid_size, ny=grid_size, dx=dx, dy=dy, dt=dt_safe,
            k=k_val, rho=2500.0, cp=700.0, T_amb=T_amb, h_conv=h_conv,
        )

    # Run simulation
    T0 = np.full((grid_size, grid_size), T_init_val)
    source_mask = mask_cell  # heat generated only in battery cells
    trajectory = solver.solve(T0, n_steps=n_steps, q0=q0, source_mask=source_mask)

    # Build spatially constant physics fields for input channels
    k_field = np.full((grid_size, grid_size), k_val)
    q_field = np.full((grid_size, grid_size), q0 / 1e6)    # normalise for NN
    h_field = np.full((grid_size, grid_size), h_conv / 500.0)  # normalise

    # Extract consecutive-step pairs as training samples
    for t_idx in range(trajectory.shape[0] - 1):
        T_curr = trajectory[t_idx]
        T_next = trajectory[t_idx + 1]
        delta_T = T_next - T_curr

        # Normalise temperature to [0, 1] range based on trajectory stats
        T_min, T_max = trajectory.min(), trajectory.max()
        T_range = T_max - T_min if T_max > T_min else 1.0
        T_norm = (T_curr - T_min) / T_range

        # Assemble 9-channel input: [T, mask_cell, mask_cool, mask_ins, k, q, h, sdf_cell, sdf_cool]
        inp = np.stack([
            T_norm,
            mask_cell,
            mask_coolant,
            mask_insulation,
            k_field / 5.0,         # normalise k to ~[0, 1]
            q_field,               # already normalised
            h_field,               # already normalised
            sdf_cell,
            sdf_coolant,
        ], axis=0)  # (9, H, W)

        all_inputs.append(inp)
        all_targets.append(delta_T[np.newaxis, :, :])  # (1, H, W)

# Convert to tensors
X = torch.tensor(np.array(all_inputs), dtype=torch.float32)
Y = torch.tensor(np.array(all_targets), dtype=torch.float32)
print(f"\nTotal samples: {X.shape[0]}")
print(f"Input shape:  {X.shape}  (N, 9, {grid_size}, {grid_size})")
print(f"Target shape: {Y.shape}  (N, 1, {grid_size}, {grid_size})")
print(f"Target delta_T range: [{Y.min():.6f}, {Y.max():.6f}]")

# Split into train (35 trajectories), val (10), test (5)
# Each trajectory contributes n_steps samples
samples_per_traj = n_steps  # trajectory length - 1, but solve returns n_steps+1 frames
n_train_traj, n_val_traj, n_test_traj = 35, 10, 5
n_train = n_train_traj * samples_per_traj
n_val = n_val_traj * samples_per_traj
n_test = n_test_traj * samples_per_traj

X_train, Y_train = X[:n_train], Y[:n_train]
X_val, Y_val = X[n_train:n_train + n_val], Y[n_train:n_train + n_val]
X_test, Y_test = X[n_train + n_val:n_train + n_val + n_test], Y[n_train + n_val:n_train + n_val + n_test]

print(f"\nTrain: {X_train.shape[0]} samples ({n_train_traj} trajectories)")
print(f"Val:   {X_val.shape[0]} samples ({n_val_traj} trajectories)")
print(f"Test:  {X_test.shape[0]} samples ({n_test_traj} trajectories)")

# Create DataLoaders
batch_size = 8
train_dataset = TensorDataset(X_train, Y_train)
val_dataset = TensorDataset(X_val, Y_val)
test_dataset = TensorDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nDataLoaders created (batch_size={batch_size}):")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

## 2. Model Architecture
Create the Physics-Conditioned U-Net and inspect its structure.

In [ ]:
model = PCUNet(in_channels=9, out_channels=1, base_features=16, num_levels=3, dropout_rate=0.1).to(device)
print(f"Model parameters: {model.count_parameters():,}")
print(f"Model config: {model.get_config()}")

## 3. Physics-Informed Loss Function
Set up the multi-component loss: data loss + PDE residual + boundary conditions + consistency.

In [ ]:
# Physics-informed loss with grid/material parameters matching our data
loss_fn = PhysicsInformedLoss(
    lambda_data=1.0,
    lambda_pde=0.1,
    lambda_bc=0.1,
    lambda_consistency=0.05,
    dx=dx,
    dy=dy,
    dt=dt,
    rho=2500.0,
    cp=700.0,
    T_amb=298.15,
)

# AdamW optimizer with weight decay for regularisation
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

# Total epochs across all three phases
phase_epochs = [10, 15, 10]  # demo: short phases
total_epochs = sum(phase_epochs)

# Cosine annealing scheduler over all phases
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_epochs,
    eta_min=1e-6,
)

# GradNorm for adaptive loss balancing (4 loss terms)
grad_norm = GradNorm(n_tasks=4, alpha=1.5, lr=0.025)

print(f"Optimizer: AdamW (lr=1e-3, weight_decay=1e-5)")
print(f"Scheduler: CosineAnnealingLR (T_max={total_epochs}, eta_min=1e-6)")
print(f"Loss function: PhysicsInformedLoss (data + PDE + BC + consistency)")
print(f"GradNorm: n_tasks=4, alpha=1.5")
print(f"Phase schedule: {phase_epochs} => {total_epochs} total epochs")

## 4. Three-Phase Training
- **Phase 1** (data-only): Learn basic input-output mapping
- **Phase 2** (physics-informed): Add PDE and BC constraints
- **Phase 3** (consistency): Add multi-step consistency loss

In [ ]:
from tqdm.auto import tqdm

# Training history
history = {
    "train_loss": [],
    "val_loss": [],
    "data_loss": [],
    "pde_loss": [],
    "bc_loss": [],
    "consistency_loss": [],
    "lr": [],
    "phase": [],
}

# Logger for tracking (writes CSV to logs/ directory)
log_dir = Path("../logs/notebook_training")
train_logger = TrainingLogger(log_dir)

# Determine phase boundaries
phase_boundaries = [0]  # epoch indices where each phase starts
for pe in phase_epochs:
    phase_boundaries.append(phase_boundaries[-1] + pe)

print(f"Phase boundaries (epoch): {phase_boundaries}")
print(f"Phase 1 (data-only):      epochs 1-{phase_boundaries[1]}")
print(f"Phase 2 (physics):        epochs {phase_boundaries[1]+1}-{phase_boundaries[2]}")
print(f"Phase 3 (consistency):    epochs {phase_boundaries[2]+1}-{phase_boundaries[3]}")
print()

global_epoch = 0

for phase_idx, n_phase_epochs in enumerate(phase_epochs, start=1):
    print(f"{'='*60}")
    print(f"Phase {phase_idx}: {['Data-only', 'Physics-informed', 'Consistency'][phase_idx-1]}")
    print(f"{'='*60}")

    # Reset GradNorm at phase transitions so it re-calibrates
    if phase_idx >= 2:
        grad_norm.reset()

    for epoch_in_phase in range(n_phase_epochs):
        global_epoch += 1
        model.train()

        epoch_train_loss = 0.0
        epoch_data_loss = 0.0
        epoch_pde_loss = 0.0
        epoch_bc_loss = 0.0
        epoch_consist_loss = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {global_epoch:3d} (P{phase_idx})", leave=False)
        for batch_x, batch_y in pbar:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            # Forward pass
            pred = model(batch_x)

            # Extract physics fields from input for PDE/BC losses
            T_input = batch_x[:, 0:1, :, :]   # current temperature (B, 1, H, W)
            k_field = batch_x[:, 4:5, :, :]    # conductivity channel
            q_field = batch_x[:, 5:6, :, :]    # heat source channel
            h_field = batch_x[:, 6:7, :, :]    # convection coeff channel

            # Cache params on model for consistency loss (channels 1-8)
            model._cached_params = batch_x[:, 1:, :, :]

            # Compute composite loss for current phase
            total_loss, loss_dict = loss_fn(
                pred=pred,
                target=batch_y,
                T_input=T_input,
                k_field=k_field,
                q_field=q_field,
                h_field=h_field,
                model=model if phase_idx >= 3 else None,
                phase=phase_idx,
            )

            # Backward pass and optimise
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # Accumulate metrics
            epoch_train_loss += loss_dict["total_loss"]
            epoch_data_loss += loss_dict.get("data_loss", 0.0)
            epoch_pde_loss += loss_dict.get("pde_loss", 0.0)
            epoch_bc_loss += loss_dict.get("bc_loss", 0.0)
            epoch_consist_loss += loss_dict.get("consistency_loss", 0.0)
            n_batches += 1

            pbar.set_postfix(loss=f"{loss_dict['total_loss']:.6f}")

        # Clear cached params
        model._cached_params = None

        # Average training metrics
        avg_train_loss = epoch_train_loss / max(n_batches, 1)
        avg_data_loss = epoch_data_loss / max(n_batches, 1)
        avg_pde_loss = epoch_pde_loss / max(n_batches, 1)
        avg_bc_loss = epoch_bc_loss / max(n_batches, 1)
        avg_consist_loss = epoch_consist_loss / max(n_batches, 1)

        # Validation
        model.eval()
        val_loss_sum = 0.0
        val_batches = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                pred = model(batch_x)
                val_mse = nn.functional.mse_loss(pred, batch_y).item()
                val_loss_sum += val_mse
                val_batches += 1
        avg_val_loss = val_loss_sum / max(val_batches, 1)

        # Step scheduler
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        # Record history
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["data_loss"].append(avg_data_loss)
        history["pde_loss"].append(avg_pde_loss)
        history["bc_loss"].append(avg_bc_loss)
        history["consistency_loss"].append(avg_consist_loss)
        history["lr"].append(current_lr)
        history["phase"].append(phase_idx)

        # Log to CSV
        train_logger.log_epoch(global_epoch, {
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "data_loss": avg_data_loss,
            "pde_loss": avg_pde_loss,
            "bc_loss": avg_bc_loss,
            "consistency_loss": avg_consist_loss,
            "lr": current_lr,
            "phase": phase_idx,
        })

        # Print epoch summary every 5 epochs or at phase end
        if global_epoch % 5 == 0 or epoch_in_phase == n_phase_epochs - 1:
            print(f"  Epoch {global_epoch:3d} | "
                  f"train_loss={avg_train_loss:.6f} | "
                  f"val_loss={avg_val_loss:.6f} | "
                  f"lr={current_lr:.2e}")

print(f"\nTraining complete: {global_epoch} epochs across 3 phases.")
print(f"Final train_loss: {history['train_loss'][-1]:.6f}")
print(f"Final val_loss:   {history['val_loss'][-1]:.6f}")

## 5. Training Loss Curves
Visualize how losses evolve during training.

In [ ]:
def moving_average(data, window=3):
    """Compute a simple moving average with the given window size."""
    if len(data) < window:
        return data
    cumsum = np.cumsum(data, dtype=float)
    cumsum[window:] = cumsum[window:] - cumsum[:-window]
    # First (window-1) entries: use expanding window
    result = np.empty(len(data))
    for i in range(window - 1):
        result[i] = cumsum[i] / (i + 1)
    result[window - 1:] = cumsum[window - 1:] / window
    return result

epochs = np.arange(1, total_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: total train/val loss ---
ax = axes[0]
ax.plot(epochs, history["train_loss"], alpha=0.3, color="tab:blue", label="Train (raw)")
ax.plot(epochs, history["val_loss"], alpha=0.3, color="tab:orange", label="Val (raw)")
ax.plot(epochs, moving_average(history["train_loss"], window=3),
        color="tab:blue", linewidth=2, label="Train (smoothed)")
ax.plot(epochs, moving_average(history["val_loss"], window=3),
        color="tab:orange", linewidth=2, label="Val (smoothed)")

# Mark phase transitions with vertical dashed lines
for boundary, label in zip(phase_boundaries[1:-1], ["Phase 1->2", "Phase 2->3"]):
    ax.axvline(x=boundary + 0.5, color="gray", linestyle="--", alpha=0.7, linewidth=1.5)
    ax.text(boundary + 0.7, ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 0 else 0.95,
            label, fontsize=9, color="gray", va="top")

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Total Training & Validation Loss")
ax.legend(fontsize=9)
ax.set_yscale("log")

# --- Right panel: individual loss components ---
ax = axes[1]
ax.plot(epochs, moving_average(history["data_loss"], window=3),
        label="Data loss", linewidth=2)
ax.plot(epochs, moving_average(history["pde_loss"], window=3),
        label="PDE loss", linewidth=2)
ax.plot(epochs, moving_average(history["bc_loss"], window=3),
        label="BC loss", linewidth=2)
ax.plot(epochs, moving_average(history["consistency_loss"], window=3),
        label="Consistency loss", linewidth=2)

for boundary, label in zip(phase_boundaries[1:-1], ["Phase 1->2", "Phase 2->3"]):
    ax.axvline(x=boundary + 0.5, color="gray", linestyle="--", alpha=0.7, linewidth=1.5)

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (component)")
ax.set_title("Loss Components by Training Phase")
ax.legend(fontsize=9)
ax.set_yscale("log")

plt.tight_layout()
plt.show()

## 6. Validation Results
Evaluate the trained model on the validation set.

In [ ]:
model.eval()

all_preds = []
all_trues = []

with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        pred = model(batch_x)
        all_preds.append(pred.cpu())
        all_trues.append(batch_y.cpu())

preds = torch.cat(all_preds, dim=0)  # (N_val, 1, H, W)
trues = torch.cat(all_trues, dim=0)  # (N_val, 1, H, W)

# Compute metrics
errors = preds - trues
mse = torch.mean(errors ** 2).item()
rmse = mse ** 0.5
mae = torch.mean(torch.abs(errors)).item()
max_error = torch.max(torch.abs(errors)).item()

# Relative error (avoid division by zero)
true_range = trues.max() - trues.min()
nrmse = rmse / (true_range.item() + 1e-8)

print("Validation Metrics")
print("=" * 40)
print(f"  MSE:         {mse:.8f}")
print(f"  RMSE:        {rmse:.8f}")
print(f"  MAE:         {mae:.8f}")
print(f"  Max Error:   {max_error:.8f}")
print(f"  NRMSE:       {nrmse:.6f} ({nrmse*100:.2f}%)")
print(f"\n  Pred range:  [{preds.min():.6f}, {preds.max():.6f}]")
print(f"  True range:  [{trues.min():.6f}, {trues.max():.6f}]")

## 7. Save Checkpoint
Save the trained model for later evaluation.

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": total_epochs,
    "config": model.get_config(),
}
ckpt_path = Path("../checkpoints")
ckpt_path.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, ckpt_path / "notebook_model.pt")
print(f"Model saved to {ckpt_path / 'notebook_model.pt'}")

## Summary
- Trained PC-U-Net with 3-phase physics-informed strategy
- Phase transitions improve physics compliance without sacrificing accuracy
- Model can be evaluated further in notebook 03